# POC-05 — Jupyter Notebook + ScenarioSession

Este notebook valida a integração entre o JupyterHub e a `ScenarioSession`.

Fluxo:

`Jupyter Notebook → ScenarioSession → ScenarioBuilder → PlanningProject → scenario.toml`

Nesta etapa não há LLM, MCP ou Planning Service.

In [1]:
from pathlib import Path
import shutil
import tomllib

from cisei_planning_sdk.project import PlanningProject
from scenario_session import ScenarioSession

print("POC-05 — imports OK")

POC-05 — imports OK


## 1. Workspace

O `cisei_workspace` é o workspace persistente do usuário.

In [2]:
WORKSPACE = Path("/home/jovyan/work/cisei_workspace")

PROJECT_NAME = "scenario_session_poc"
PROJECT_PATH = WORKSPACE / "projects" / PROJECT_NAME

GOLDEN_PATH = (
    WORKSPACE
    / "projects"
    / "example1_LTE_planning_basic"
    / "scenario.toml"
)

print("Workspace:")
print(WORKSPACE)
print()
print("Projeto:")
print(PROJECT_PATH)
print()
print("Golden reference:")
print(GOLDEN_PATH)

Workspace:
/home/jovyan/work/cisei_workspace

Projeto:
/home/jovyan/work/cisei_workspace/projects/scenario_session_poc

Golden reference:
/home/jovyan/work/cisei_workspace/projects/example1_LTE_planning_basic/scenario.toml


In [3]:
assert WORKSPACE.exists(), f"Workspace não encontrado: {WORKSPACE}"
assert GOLDEN_PATH.exists(), f"Golden reference não encontrado: {GOLDEN_PATH}"

print("Workspace e golden reference encontrados.")

Workspace e golden reference encontrados.


## 2. Carregar o cenário de referência

O cenário real `example1_LTE_planning_basic` será utilizado como referência para simular as informações fornecidas pelo usuário.

In [4]:
golden_data = tomllib.loads(
    GOLDEN_PATH.read_text(encoding="utf-8")
)

print("Seções do cenário:")

for section in golden_data:
    print(f"  - {section}")

Seções do cenário:
  - scenario
  - antennas
  - interfaces
  - devices
  - metrics
  - instances


## 3. Criar o PlanningProject e a ScenarioSession

In [5]:
if PROJECT_PATH.exists():
    shutil.rmtree(PROJECT_PATH)

project = PlanningProject(PROJECT_PATH)
session = ScenarioSession(project)

print("Projeto criado:")
print(project.path)
print()
print("ScenarioSession criada.")

Projeto criado:
/home/jovyan/work/cisei_workspace/projects/scenario_session_poc

ScenarioSession criada.


## 4. Estado inicial da sessão

In [6]:
status = session.status()

print("Estado inicial")
print("-------------")
print(f"valid:     {status['valid']}")
print(f"complete:  {status['complete']}")
print(f"confirmed: {status['confirmed']}")
print(f"missing:   {len(status['missing'])}")

print("\nCampos faltantes:")

for field in status["missing"]:
    print(f"  - {field}")

Estado inicial
-------------
valid:     True
complete:  False
confirmed: False
missing:   8

Campos faltantes:
  - scenario.working_crs
  - instances.source
  - instances.format
  - instances.profile_column
  - antennas.<at_least_one>
  - interfaces.<at_least_one>
  - devices.<at_least_one>
  - metrics.<at_least_one>


## 5. Construção incremental

Cada seção representa informações que poderiam ter sido fornecidas pelo usuário durante uma conversa.

In [7]:
for section_name, section_data in golden_data.items():
    print(f"\n=== Atualizando: {section_name} ===")

    validation = session.update({
        section_name: section_data
    })

    print(f"valid:    {validation.valid}")
    print(f"complete: {validation.complete}")
    print(f"missing:  {len(validation.missing)}")

    if validation.missing:
        print("Campos faltantes:")

        for field in validation.missing:
            print(f"  - {field}")


=== Atualizando: scenario ===
valid:    True
complete: False
missing:  7
Campos faltantes:
  - instances.source
  - instances.format
  - instances.profile_column
  - antennas.<at_least_one>
  - interfaces.<at_least_one>
  - devices.<at_least_one>
  - metrics.<at_least_one>

=== Atualizando: antennas ===
valid:    True
complete: False
missing:  6
Campos faltantes:
  - instances.source
  - instances.format
  - instances.profile_column
  - interfaces.<at_least_one>
  - devices.<at_least_one>
  - metrics.<at_least_one>

=== Atualizando: interfaces ===
valid:    True
complete: False
missing:  5
Campos faltantes:
  - instances.source
  - instances.format
  - instances.profile_column
  - devices.<at_least_one>
  - metrics.<at_least_one>

=== Atualizando: devices ===
valid:    True
complete: False
missing:  4
Campos faltantes:
  - instances.source
  - instances.format
  - instances.profile_column
  - metrics.<at_least_one>

=== Atualizando: metrics ===
valid:    True
complete: False
missing: 

## 6. Validação final

In [8]:
status = session.status()

print("Validação final")
print("---------------")
print(f"valid:     {status['valid']}")
print(f"complete:  {status['complete']}")
print(f"missing:   {status['missing']}")
print(f"confirmed: {status['confirmed']}")

assert status["valid"] is True
assert status["complete"] is True
assert status["missing"] == []
assert status["confirmed"] is False

print("\nCenário completo e válido: OK")

Validação final
---------------
valid:     True
complete:  True
missing:   []
confirmed: False

Cenário completo e válido: OK


## 7. Confirmação explícita

A confirmação representa a aceitação explícita da especificação construída.

In [9]:
validation = session.confirm()

print("Confirmação")
print("-----------")
print(f"valid:     {validation.valid}")
print(f"complete:  {validation.complete}")
print(f"confirmed: {session.confirmed}")

assert validation.valid is True
assert validation.complete is True
assert session.confirmed is True

print("\nCenário confirmado: OK")

Confirmação
-----------
valid:     True
complete:  True
confirmed: True

Cenário confirmado: OK


## 8. Geração e persistência do scenario.toml

In [10]:
toml_text = session.generate_toml()
saved_path = session.save()

print("TOML gerado:")
print(f"  caracteres: {len(toml_text)}")
print()
print("Arquivo salvo:")
print(f"  {saved_path}")

assert saved_path.exists()
assert saved_path == project.scenario_path

print("\nPersistência: OK")

TOML gerado:
  caracteres: 1844

Arquivo salvo:
  /home/jovyan/work/cisei_workspace/projects/scenario_session_poc/scenario.toml

Persistência: OK


## 9. Comparação com o golden reference

A comparação é estrutural, usando `tomllib`.

In [11]:
generated_data = tomllib.loads(
    project.scenario_path.read_text(encoding="utf-8")
)

assert generated_data == golden_data

print("=" * 60)
print("RESULTADO: IDENTICOS")
print("=" * 60)

RESULTADO: IDENTICOS


## 10. Invalidação da confirmação

Qualquer alteração após a confirmação deve exigir uma nova confirmação.

In [12]:
assert session.confirmed is True

session.update({
    "scenario": {
        "working_crs": "EPSG:31982"
    }
})

print(f"confirmed após alteração: {session.confirmed}")

assert session.confirmed is False

print("Invalidação da confirmação: OK")

confirmed após alteração: False
Invalidação da confirmação: OK


## 11. Resultado final

In [13]:
print("=" * 60)
print("POC-05 — JUPYTER + SCENARIO SESSION — TESTE PASSOU")
print("=" * 60)
print()
print(f"Workspace: {WORKSPACE}")
print(f"Projeto:   {project.path}")
print(f"Scenario:  {project.scenario_path}")
print()
print("Fluxo validado:")
print("  Jupyter Notebook")
print("       ↓")
print("  ScenarioSession")
print("       ↓")
print("  ScenarioBuilder")
print("       ↓")
print("  PlanningProject")
print("       ↓")
print("  scenario.toml")

POC-05 — JUPYTER + SCENARIO SESSION — TESTE PASSOU

Workspace: /home/jovyan/work/cisei_workspace
Projeto:   /home/jovyan/work/cisei_workspace/projects/scenario_session_poc
Scenario:  /home/jovyan/work/cisei_workspace/projects/scenario_session_poc/scenario.toml

Fluxo validado:
  Jupyter Notebook
       ↓
  ScenarioSession
       ↓
  ScenarioBuilder
       ↓
  PlanningProject
       ↓
  scenario.toml
